# hy-cuts L=4 — electric-cut O_FM(hz) at h_y = 0, 0.2, 0.4

Side-by-side Z-string order parameter along the electric cut (h_x=0.2, sweep h_z),
sign-free vs sign-full. Points are **end-of-training pooled** `O_FM_paratoric`
(`--final_eval_rounds 8`): h_y=0 from the Phase-B rerun, h_y=0.2/0.4 from the
2026-08-26 hy-cuts campaign (jobs 57623248-53 / 57623300-05) — identical 500-step
dual-basis protocol. S₂ and per-snapshot series follow from the replay jobs.

In [ ]:
# %% 1 CONFIG
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt

ROOT = Path("../../results")
HYS  = [0.0, 0.2, 0.4]                          # panel order
DIRS = {0.0: ROOT / "phaseB_rerun/up/L4",       # sign-free reference, same protocol
        0.2: ROOT / "hy_cuts_L4/up/hy0.2/L4",
        0.4: ROOT / "hy_cuts_L4/up/hy0.4/L4"}
ERR_X = 3                                       # bars x3 on order-parameter panels (house rule)
SAVE_FIGS, FIGS = False, Path("../figs")

In [ ]:
# %% 2 load: (hz, O_FM, err) per hy, from final-state JSONs
def ofm_curve(d, hy):
    rows = []
    for f in sorted(Path(d).glob("*.json")):
        j = json.loads(f.read_text())
        c, o = j["config"], j.get("observables", {})
        if abs(c.get("hy", 0.0) - hy) > 1e-9 or j.get("diverged") \
           or o.get("O_FM_paratoric") is None:
            continue
        rows.append((c["hz"], o["O_FM_paratoric"], o["O_FM_paratoric_err"]))
    hz, v, e = map(np.array, zip(*sorted(rows)))
    return hz, v, e

curves = {hy: ofm_curve(DIRS[hy], hy) for hy in HYS}
{hy: len(c[0]) for hy, c in curves.items()}

In [ ]:
# %% 3 figure: one panel per hy, sign-free curve as open reference
cpos = {0.0: 0.15, 0.2: 0.5, 0.4: 0.8}         # plasma keyed by hy
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4), sharex=True, sharey=True)
for ax, hy in zip(axes, HYS):
    if hy:
        hz0, v0, e0 = curves[0.0]
        ax.errorbar(hz0, v0, ERR_X * e0, fmt="o", mfc="none", color="0.6",
                    ms=5, capsize=2, lw=1, label="$h_y=0$ ref")
    hz, v, e = curves[hy]
    ax.errorbar(hz, v, ERR_X * e, fmt="o", color=plt.cm.plasma(cpos[hy]),
                ms=5, capsize=2, lw=1, label=f"$h_y={hy}$")
    ax.set_title(f"$h_y = {hy}$")
    ax.set_xlabel("$h_z$")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="upper left")
axes[0].set_ylabel(rf"$O_\mathrm{{FM}}$ (Z-string)  (bars $\times${ERR_X})")
fig.suptitle("3D TC, L=4 OBC, electric cut ($h_x=0.2$): transition vs $h_y$", y=1.03)
fig.tight_layout()
# if SAVE_FIGS: plt.savefig(FIGS / "hy_cuts_L4_ofm_electric.png", dpi=300, bbox_inches="tight")
plt.show()

**Reading**: O_FM rises earlier in h_z as h_y grows — the y-field destabilizes the
topological phase, shifting the electric transition down (half-max: ≈0.29 at h_y=0/0.2,
≈0.27 at h_y=0.4; barely resolvable at 0.2 on this coarse grid). Grid refinement at
h_z=0.24/0.28 and the S₂/inflection locators from the snapshot replay sharpen h_c.

## S₂-Rényi locator (snapshot replay, step-500 states)

Central-plaquette S₂ from `eval_snapshots.py --topological --fm_sector electric`
(jobs 57644883-87). Deep topological anchor = exact 3·ln2; the S₂ collapse is the
independent transition locator. No h_y=0 replay series exists (pre-`--topological`
replays), so the anchor line stands in as the reference.

In [ ]:
# %% 4 S2 vs hz from the snapshot series (last snapshot per run)
S2LN2 = 3 * np.log(2)
def s2_curve(d, hy):
    rows = []
    for f in sorted(Path(d).glob("*.snapshots.json")):
        j = json.loads(f.read_text())
        c = j.get("config", {})
        if abs(c.get("hy", 0.0) - hy) > 1e-9:
            continue
        ser = [s for s in j.get("series", []) if "error" not in s and s.get("S2") is not None]
        if not ser:
            continue
        last = ser[-1]
        rows.append((c["hz"], last["S2"], last["S2_err"]))
    hz, v, e = map(np.array, zip(*sorted(rows)))
    return hz, v, e

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharex=True, sharey=True)
for ax, hy in zip(axes, [0.2, 0.4]):
    hz, v, e = s2_curve(DIRS[hy], hy)
    ax.axhline(S2LN2, color="0.75", lw=1, ls="--", label=r"$3\ln 2$ (topo)")
    ax.errorbar(hz, v, ERR_X * e, fmt="o", color=plt.cm.plasma(cpos[hy]),
                ms=5, capsize=2, lw=1, label=f"$h_y={hy}$")
    ax.set_title(f"$h_y = {hy}$")
    ax.set_xlabel("$h_z$")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="lower left")
axes[0].set_ylabel(rf"$S_2$ (central plaquette)  (bars $\times${ERR_X})")
fig.suptitle("S$_2$ collapse along the electric cut", y=1.03)
fig.tight_layout()
# if SAVE_FIGS: plt.savefig(FIGS / "hy_cuts_L4_s2_electric.png", dpi=300, bbox_inches="tight")
plt.show()

**Reading**: S₂ sits at the topological plateau (≈3ln2) through hz≈0.22, then collapses —
midpoint between 0.26–0.30 at h_y=0.2 vs 0.22–0.26 at h_y=0.4, consistent with the O_FM
shift of h_c down in h_z with growing h_y. §A drift check: last-100-step O_FM drift ≤0.04
at every point — all 12 runs converged, no 750-step extensions needed. Error-bar caveat:
S₂ at h_y≠0 has no phase-coherence diagnostic yet (audit MEDIUM) — treat bars as optimistic.

## Right/magnetic cut (h_z=0.1, sweep h_x): hysteresis loops

§B warm chains: **up** = topological anchor (h_x=0.6) carried up, **dn** = trivial anchor
(h_x=1.25) carried down; 0.05 steps, dt=0.005, 200 steps/link. Rows: energy per spin
(branches overlap at this scale), the loop opening ΔE/N = (E_up−E_dn)/N, the magnetic
X-membrane order parameter, and central-plaquette S₂ (from the post-hoc
`topological_observables` pass, `results/hy_cuts_L4/right_topo_hy*.json`).
No link crashed or shed anywhere: no spinodal inside the window at L=4.

In [ ]:
# %% 5 right-cut branch loader
N_SPINS = 144                                     # L=4 OBC: 3L^3 - 3L^2
def right_branches(hy):
    d = ROOT / f"hy_cuts_L4/right/hy{hy}/L4"
    up, dn = {}, {}
    for f in sorted(d.glob("*_k3*.json")):
        if f.name.endswith(("snapshots.json", "curve.json")):
            continue
        j = json.loads(f.read_text()); c, o = j["config"], j.get("observables", {})
        if c.get("hy", 0) < 0 or o.get("E0") is None:
            continue
        row = (o["E0"], o["E_err"], o.get("O_FM_membrane_R1"), o.get("O_FM_membrane_R1_err"))
        b = up if f.stem.endswith("_up") else dn if f.stem.endswith("_dn") else \
            (up if c["hx"] < 1.0 else dn)         # anchors seed their chain
        b[c["hx"]] = row
    return up, dn

def topo_rows(hy):
    f = ROOT / f"hy_cuts_L4/right_topo_hy{hy}.json"
    if not f.exists():
        return {}
    out = {}
    for r in json.loads(f.read_text()):
        if "error" in r or r.get("hy", 0) < 0 or r.get("S2") is None:
            continue
        tag = "up" if r["name"].endswith("_up") else "dn" if r["name"].endswith("_dn") else \
              ("up" if r["hx"] < 1.0 else "dn")
        out.setdefault(tag, {})[r["hx"]] = (r["S2"], r["S2_err"], r["O_FM"], r["O_FM_err"])
    return out

R = {hy: right_branches(hy) for hy in [0.2, 0.4]}
T = {hy: topo_rows(hy) for hy in [0.2, 0.4]}
{hy: (len(R[hy][0]), len(R[hy][1]), {t: len(v) for t, v in T[hy].items()}) for hy in R}

In [ ]:
# %% 6 hysteresis figure: rows = E/N, dE/N, membrane O_FM, S2; cols = hy
def _xy(d, i, ie=None):
    hx = np.array(sorted(d)); v = np.array([d[x][i] for x in hx])
    e = np.array([d[x][ie] for x in hx]) if ie is not None else None
    return hx, v, e

fig, axes = plt.subplots(4, 2, figsize=(9, 11), sharex=True)
for col, hy in enumerate([0.2, 0.4]):
    up, dn = R[hy]; c = plt.cm.plasma(cpos[hy])
    style = {"up": dict(fmt="o-", color=c, ms=5, capsize=2, lw=1),
             "dn": dict(fmt="s--", color=c, mfc="none", ms=5, capsize=2, lw=1)}
    for tag, d in (("up", up), ("dn", dn)):
        hx, E, Ee = _xy(d, 0, 1)
        axes[0, col].errorbar(hx, E / N_SPINS, Ee / N_SPINS, label=f"{tag}-chain", **style[tag])
        hx, O, Oe = _xy(d, 2, 3)
        axes[2, col].errorbar(hx, O, ERR_X * Oe, label=tag, **style[tag])
    both = sorted(set(up) & set(dn))
    dE = np.array([up[x][0] - dn[x][0] for x in both])
    dEe = np.array([np.hypot(up[x][1], dn[x][1]) for x in both])
    axes[1, col].errorbar(both, dE / N_SPINS, dEe / N_SPINS, fmt="o", color=c, ms=5, capsize=2)
    axes[1, col].axhline(0, color="0.8", lw=1)
    if T[hy]:
        for tag in ("up", "dn"):
            hx, s2, s2e = _xy(T[hy][tag], 0, 1)
            axes[3, col].errorbar(hx, s2, ERR_X * s2e, label=tag, **style[tag])
        axes[3, col].axhline(3 * np.log(2), color="0.75", lw=1, ls="--")
    else:
        axes[3, col].text(0.5, 0.5, "S2 eval pending", transform=axes[3, col].transAxes,
                          ha="center", color="0.5")
    axes[0, col].set_title(f"$h_y = {hy}$")
    axes[3, col].set_xlabel("$h_x$")
for ax, lab in zip(axes[:, 0], ["$E/N$", r"$\Delta E/N$ (up$-$dn)",
                                rf"$O_\mathrm{{FM}}$ membrane R1 (bars $\times${ERR_X})",
                                rf"$S_2$ (bars $\times${ERR_X})"]):
    ax.set_ylabel(lab)
for ax in axes.ravel():
    ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Right cut ($h_z=0.1$): up/dn branch loops", y=0.995)
fig.tight_layout()
# if SAVE_FIGS: plt.savefig(FIGS / "hy_cuts_L4_right_hysteresis.png", dpi=300, bbox_inches="tight")
plt.show()

**Reading**: branches merge above h_x≈0.9 (ΔE/N → 0 within errors, identical membrane
values) — no persistent L=4 hysteresis; the loop opens only near h_x=0.8 where the
up-carried state lags (ΔE/N ≈ −0.002, membrane 0.27 vs 0.42), matching the sign-free
rerun's behavior at the same point. The winner-branch curves are h_y-insensitive
(m(h_x=1.0) = 0.892/0.890/0.894 at h_y=0/0.2/0.4): the first-order feature does not
move measurably with h_y at this size, unlike the electric cut.